# Détection d'attaques inconnues par Anomaly Detection — UNSW-NB15

**Question centrale** : *peut-on détecter des attaques inconnues sans disposer d'exemples de ces attaques pendant l'entraînement ?*

Un SOC veut savoir si l'apprentissage du **comportement normal** suffit à repérer des événements inconnus. On apprend donc uniquement sur du trafic normal, puis on mesure ce que les modèles signalent comme anormal.

### Hypothèses testées

| | Hypothèse | Expérience |
|---|---|---|
| **H1** | Les méthodes de détection d'anomalies auront des compromis différents entre détection et faux positifs | Expérience 1 |
| **H2** | Plus le jeu d'entraînement « normal » est contaminé par des attaques, moins les modèles distinguent normal et attaque | Expérience 2 |
| **H3** | Certaines méthodes d'anomalies seront moins précises globalement qu'un classifieur supervisé, mais plus utiles sur certaines familles inconnues | Expérience 3 |

### Plan du notebook

| Section | Contenu | Phase du sujet |
|---|---|---|
| 1 | Configuration, graines, versions des bibliothèques | Reproductibilité |
| 2 | Chargement et EDA ciblée | Phase 1 |
| 3 | Protocole : découpage, règle de seuil, métriques | Phase 2 |
| 4 | Pipeline de prétraitement, baseline et détecteurs | Phase 2 |
| 5 | Expérience 1 : IF, LOF, One-Class SVM + analyse des FP/FN | Phase 3 |
| 6 | Expérience 2 : contamination 0 → 10 % | Phase 4 |
| 7 | Expérience 3 : simulation zero-day vs classifieur supervisé | Phase 5 |
| 8 | Explicabilité et robustesse | Phase 6 |
| 9 | Questions d'analyse Q1 à Q4 | Phase 7 |
| 10 | Conclusion et limites générales | Phase 7 |

Chaque expérience suit la structure demandée : **question de recherche, hypothèse, protocole, résultats, analyse, interprétation cyber, limites**.
Dans les analyses, on distingue explicitement **observation expérimentale**, **interprétation ML**, **interprétation cybersécurité** et **limite**.

> **Important.** Les cellules marquées ✍️ doivent être complétées avec **vos** chiffres après exécution. Les cellules « Observation automatique » impriment les résultats mesurés sur lesquels appuyer chaque conclusion.

## Pourquoi cette approche ?

### Pourquoi la détection d'anomalies ?
Un classifieur supervisé apprend à reconnaître les attaques **qu'on lui a montrées**. Il ne peut pas, par construction, apprendre une attaque qui n'existait pas au moment de l'entraînement. La détection d'anomalies inverse le problème : on modélise ce qui est **normal**, et tout ce qui s'en écarte devient suspect. C'est la seule famille de méthodes qui peut, en principe, répondre à la question centrale. Le prix à payer est connu : tout ce qui est inhabituel n'est pas malveillant, d'où un risque de fausses alertes, que l'on mesure tout au long du notebook.

### Pourquoi UNSW-NB15 ?
- Il est plus récent et plus réaliste que KDD Cup 99 / NSL-KDD, longtemps utilisés mais jugés obsolètes.
- Il fournit la **famille** de chaque attaque (`attack_cat`, 9 familles). C'est indispensable pour l'expérience 3, qui retire des familles entières de l'apprentissage.
- Il propose un **train et un test officiels séparés** par ses auteurs : on évite de mesurer la performance sur des flux presque identiques à ceux de l'entraînement.
- Limite assumée : le trafic est **généré en laboratoire** (outil IXIA PerfectStorm), avec des artefacts connus (TTL, section 8.2).

## Registre des choix méthodologiques

Ce tableau rassemble **toutes les décisions** prises dans le notebook, leur justification et l'alternative écartée. Il sert de support pour défendre la méthode en soutenance. Chaque choix est détaillé dans la section indiquée.

| Choix | Valeur retenue | Justification | Alternative écartée (et pourquoi) | Section |
|---|---|---|---|---|
| Train / test | Fichiers officiels, le plus gros = train | Respecte le découpage des auteurs ; le test est réellement séparé de l'entraînement | Découper un seul fichier au hasard : flux du test presque identiques à ceux du train → scores trop optimistes | 2 |
| Validation | 20 % du train officiel, stratifiée par famille | ~35 000 flux dont ~11 000 normaux : un quantile à 1 % est estimé sur ~110 flux, donc stable. La stratification garde les familles rares dans les mêmes proportions | 10 % : seuils plus instables. Validation croisée : coût ×5 avec One-Class SVM, pour un gain limité | 3 |
| Doublons | Retirés **seulement** de l'entraînement des détecteurs | LOF calcule des distances entre voisins : des flux identiques donnent des distances nulles et des densités infinies. Validation et test gardent les doublons pour refléter le vrai trafic | Dédoublonner partout : fausserait les taux mesurés (un flux répété 100 fois compte 100 fois en réalité) | 3 |
| Taille d'entraînement | `N_FIT` = 20 000 flux normaux, identique pour tous | One-Class SVM a un coût d'entraînement entre quadratique et cubique en nombre de points, LOF fait une recherche de voisins par flux scoré. 20 000 flux décrivent bien le normal et les 60 entraînements de l'expérience 2 restent faisables | Tout le pool normal (~45 000 flux) : One-Class SVM trop lent. Taille différente par modèle : comparaison inéquitable | 1, 3 |
| Prétraitement | Appris sur l'entraînement de chaque modèle (`Pipeline`) | Exigence anti-fuite du sujet. En détection d'anomalies, standardiser sur le normal revient à **définir** ce qu'est un flux normal | Standardiser sur tout le dataset : le test influencerait les transformations (fuite) | 4.1 |
| Valeurs extrêmes | Écrêtage au 99e percentile | Protège les moyennes, écarts-types et distances de quelques valeurs démesurées, tout en gardant 1 % de valeurs extrêmes, qui sont souvent **le signal** recherché | 95e percentile (notebook d'origine) : écrase 5 % des valeurs, donc efface des anomalies. Aucun écrêtage : quelques valeurs énormes dominent l'échelle | 4.1 |
| Asymétrie | `log1p` sur les variables continues | Octets, paquets, durées s'étalent sur plusieurs ordres de grandeur ; le log les ramène à une échelle comparable. `log1p` accepte la valeur 0 | Box-Cox : refuse les zéros. `QuantileTransformer` : uniformise tout et efface les écarts entre valeurs extrêmes | 4.1 |
| Mise à l'échelle | `StandardScaler` sur toutes les variables numériques | LOF, One-Class SVM et la baseline reposent sur des distances : sans mise à l'échelle, la variable aux plus grandes valeurs décide seule | `RobustScaler` : possible, mais l'écrêtage et le log ont déjà neutralisé les extrêmes | 4.1 |
| Catégorielles | One-hot des 5 modalités les plus fréquentes + « rare » | Le top 5 couvre la grande majorité des flux ; peu de colonnes en plus, ce qui protège les méthodes à distance du fléau de la dimension. Une modalité jamais vue va dans « rare » | One-hot complet (> 130 colonnes pour `proto`, presque toutes vides). Encodage ordinal : crée un ordre artificiel entre protocoles, qui fausse les distances | 4.1 |
| Baseline | Écart moyen en z-score au trafic normal | Le détecteur le plus simple possible : si les modèles avancés ne font pas mieux, ils n'apportent rien | Baseline « aucune alerte » : triviale, rappel nul, n'apprend rien | 4.2 |
| Détecteurs | Isolation Forest, LOF, One-Class SVM | Imposés par le sujet ; ils reposent sur trois principes différents (isolation, densité locale, frontière), ce qui est exactement ce que H1 interroge | Autoencodeur : plus de réglages et de temps ; proposé en piste | 4.2 |
| Hyperparamètres | Fixés a priori (valeurs usuelles) | Évite d'optimiser sur le test et de favoriser un modèle. Optimiser demanderait des attaques étiquetées, ce qui contredit l'hypothèse « pas d'exemples d'attaques » | Recherche sur grille avec les labels de validation : performance plus haute mais irréaliste dans le scénario étudié | 4.2 |
| Seuil d'alerte | Quantile `1 − TARGET_FPR` des scores des **normaux de validation** | N'utilise aucune attaque ; met tous les modèles au **même budget de fausses alertes** ; correspond au raisonnement d'un SOC | Paramètre `contamination` d'IF ou seuil 0 d'OC-SVM : non comparables entre modèles. Seuil maximisant le F1 avec les labels : utilise des attaques, donc optimiste | 3.2 |
| FPR visé | 5 %, complété par 1, 2 et 10 % | 5 % = `nu` du One-Class SVM, et produit assez d'alertes pour mesurer un rappel fiable. Les autres niveaux montrent le compromis complet (H1, Q3) | 1 % seul : rappels très faibles, comparaisons bruitées | 1, 5.4 |
| Sélection du modèle analysé | Meilleure **PR-AUC de validation** | Indépendante du seuil, centrée sur la classe attaque, adaptée au déséquilibre ; mesurée hors test | Accuracy : trompeuse. ROC-AUC : paraît bonne dès qu'il y a beaucoup de normaux. Choix sur le test : interdit par le sujet | 5.1 |
| Critère H1 | Classement qui change selon le FPR **ou** écart de rappel > 5 points | Fixé **avant** les résultats ; 5 points de rappel représentent une différence concrète pour un SOC | Test statistique formel : un seul jeu de test, peu de modèles | 5.4 |
| Contamination | 0, 1, 2, 5, 10 % ; 3 répétitions | Niveaux imposés ; 3 tirages = minimum pour estimer une dispersion dans le temps disponible | 1 tirage : impossible de distinguer un effet du hasard | 6 |
| Seuil en Exp. 2 | Fixé sur des normaux **propres** | Isole l'effet de la contamination **sur le modèle** | Validation contaminée : mélange deux effets (modèle et seuil) ; noté en limite | 6 |
| Mesure de tendance | Δ entre 0 et 10 % + corrélation de Spearman | Spearman mesure une tendance monotone sans supposer de relation linéaire, adaptée à 5 niveaux | Régression linéaire : suppose une relation linéaire non justifiée | 6 |
| Familles inconnues | Reconnaissance, Backdoor, Shellcode, Worms | Comportements variés (balayage, porte dérobée, charge malveillante, propagation) ; Reconnaissance a assez de flux de test pour une mesure fiable. Les grandes familles (Generic, Exploits) restent connues pour que le supervisé reste réaliste | Une seule famille : conclusion trop dépendante de ce choix. Retirer les grandes familles : le supervisé perdrait l'essentiel de ses exemples | 7 |
| Modèle supervisé | Random Forest, 200 arbres, `class_weight="balanced_subsample"` | Meilleur compromis dans le notebook supervisé précédent ; peu sensible aux hyperparamètres ; référence classique en détection d'intrusion. Les poids de classe compensent le déséquilibre normal/attaque dans chaque arbre | Gradient boosting : souvent un peu meilleur, mais plus de réglages | 7 |
| Seuil du supervisé | Même règle que les détecteurs | Comparaison **au même FPR** : seule façon équitable de comparer les rappels | Seuil 0,5 par défaut : FPR différent, donc rappels non comparables | 7 |
| Explicabilité | Importance par permutation, sur la validation | S'applique aux 5 modèles avec la même méthode ; porte sur les variables d'origine ; calculée hors test | SHAP : rapide et exact pour les arbres, mais très coûteux pour LOF et One-Class SVM (`KernelExplainer`) | 8.1 |
| Robustesse | Isolation Forest sans les TTL | Teste un **artefact connu du dataset** plutôt qu'un modèle ; IF est le plus rapide à réentraîner | Refaire toutes les expériences sans TTL : coût élevé, proposé en piste | 8.2 |
| Volume pour Q3 | 1 million de flux normaux par jour | Ordre de grandeur **hypothétique** pour traduire un FPR en nombre d'alertes ; paramétrable | — | 9 |

## 1. Configuration et reproductibilité

Tous les paramètres des expériences sont regroupés ici pour être visibles et modifiables à un seul endroit.

| Paramètre | Valeur | Rôle et justification |
|---|---|---|
| `RANDOM_STATE` | 42 | Graine commune (découpages, échantillonnages, modèles) : relancer le notebook donne les mêmes résultats. La valeur elle-même n'a pas d'importance, seul le fait de la fixer compte |
| `VAL_SIZE` | 0,2 | Assez de flux normaux (~11 000) pour estimer un seuil à 1 % de FPR sur une centaine de flux |
| `N_FIT` | 20 000 | Même taille d'entraînement pour tous les modèles (équité) ; limitée par le coût de One-Class SVM et LOF |
| `TARGET_FPR` | 5 % | Budget de fausses alertes servant à fixer le seuil ; égal au `nu` du One-Class SVM |
| `FPR_LEVELS` | 1, 2, 5, 10 % | Budgets testés pour décrire le compromis détection / fausses alertes (H1, Q3) |
| `CONTAMINATION_LEVELS` | 0 à 10 % | Niveaux imposés par le sujet |
| `N_SEEDS` | 3 | Répétitions de l'expérience 2 : minimum pour mesurer une dispersion |
| `UNKNOWN_FAMILIES` | 4 familles | Familles retirées de l'apprentissage pour l'expérience 3 (choix justifié en section 7) |
| `CLIP_QUANTILE` | 0,99 | Écrêtage des valeurs extrêmes, plus doux que le 95e percentile de l'original |
| `VOLUME_NORMAL_PAR_JOUR` | 1 000 000 | Hypothèse de volume pour traduire un FPR en fausses alertes par jour (Q3) |
| `FAST_MODE` | désactivé | Version rapide pour vérifier que le notebook tourne ; résultats non exploitables pour le rapport |

Chaque valeur est justifiée plus en détail dans le **registre des choix** ci-dessus.

In [ ]:
import os
import sys
import time
import warnings
import platform

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import sklearn
from scipy.stats import spearmanr

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, PrecisionRecallDisplay, RocCurveDisplay,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

# ---------------- Paramètres ----------------
RANDOM_STATE = 42
DATA_DIR = os.environ.get("UNSW_DATA_DIR", "/kaggle/input/unsw-nb15")
TRAIN_FILE = "UNSW_NB15_training-set.csv"
TEST_FILE = "UNSW_NB15_testing-set.csv"
OUT_DIR = "results"                    # tableaux et figures pour le rapport

FAST_MODE = os.environ.get("UNSW_FAST", "0") == "1"

VAL_SIZE = 0.2                         # part du train officiel gardée pour la validation
N_FIT = 2_000 if FAST_MODE else 20_000 # flux d'entraînement par détecteur
TARGET_FPR = 0.05                      # FPR visé sur les flux normaux de validation
FPR_LEVELS = [0.01, 0.02, 0.05, 0.10]  # niveaux pour l'analyse du compromis (H1, Q3)
CONTAMINATION_LEVELS = [0.0, 0.01, 0.02, 0.05, 0.10]
N_SEEDS = 1 if FAST_MODE else 3        # répétitions de l'expérience 2
UNKNOWN_FAMILIES = ["Reconnaissance", "Backdoor", "Shellcode", "Worms"]
CLIP_QUANTILE = 0.99
VOLUME_NORMAL_PAR_JOUR = 1_000_000     # hypothèse de volume pour traduire un FPR en alertes/jour (Q3)

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(RANDOM_STATE)

def save_fig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), bbox_inches="tight")

def save_table(df, name):
    df.to_csv(os.path.join(OUT_DIR, f"{name}.csv"))

# ---------------- Versions (à reporter dans le README / rapport) ----------------
versions = {
    "Python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__, "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__, "seaborn": sns.__version__,
}
print("Mode rapide :", FAST_MODE)
pd.Series(versions, name="version").to_frame()

## 2. Données et EDA ciblée

**Dataset** : UNSW-NB15, version partitionnée (UNSW Canberra, 2015). Chaque ligne est un **flux réseau** décrit par 42 variables, avec `label` (0 normal / 1 attaque) et `attack_cat` (famille d'attaque).

Sur Kaggle, les deux fichiers sont souvent **inversés** (le fichier *training* contient 82 332 lignes). On prend le plus gros comme train officiel (175 341 lignes), ce qui correspond au découpage des auteurs.

L'EDA est **ciblée** sur les questions qui conditionnent le protocole :
1. Combien de flux normaux pour apprendre le comportement normal ?
2. Quelles familles d'attaques existent et en quelle quantité dans le test (assez d'exemples pour l'expérience 3) ?
3. Y a-t-il des doublons qui fausseraient un apprentissage par voisinage (LOF) ?
4. Les attaques se distinguent-elles visiblement du normal sur quelques variables clés ?

In [1]:
df_a = pd.read_csv(os.path.join(DATA_DIR, TRAIN_FILE))
df_b = pd.read_csv(os.path.join(DATA_DIR, TEST_FILE))
if len(df_a) >= len(df_b):
    train_df, test_df = df_a.copy(), df_b.copy()
else:
    train_df, test_df = df_b.copy(), df_a.copy()
    print("Fichiers inversés détectés : le fichier 'testing' sert de train officiel.")

for df in (train_df, test_df):
    df["attack_cat"] = df["attack_cat"].astype(str).str.strip()

FEATURES = [c for c in train_df.columns if c not in ("id", "attack_cat", "label")]
CAT_COLS = ["proto", "service", "state"]

print(f"Train officiel : {len(train_df):>7} flux | Test officiel : {len(test_df):>7} flux | {len(FEATURES)} variables")
print("Valeurs manquantes :", int(train_df.isna().sum().sum()), "/", int(test_df.isna().sum().sum()))

NameError: name 'pd' is not defined

In [ ]:
fam = pd.DataFrame({
    "train": train_df["attack_cat"].value_counts(),
    "test": test_df["attack_cat"].value_counts(),
}).fillna(0).astype(int).sort_values("train", ascending=False)
fam["inconnue (Exp. 3)"] = fam.index.isin(UNKNOWN_FAMILIES)

dup_normal = train_df[train_df["label"] == 0].duplicated(subset=FEATURES).mean()
print(f"Part de flux normaux dupliqués dans le train : {dup_normal:.1%}")

fig, ax = plt.subplots(figsize=(8, 4.5))
fam[["train", "test"]].plot(kind="barh", logx=True, ax=ax)
ax.invert_yaxis()
ax.set_title("Flux par famille (échelle log)")
ax.set_xlabel("Nombre de flux")
plt.tight_layout(); save_fig(fig, "eda_familles"); plt.show()
fam

In [ ]:
KEY_VARS = [c for c in ["dur", "sbytes", "dbytes", "rate", "sttl", "smean"] if c in FEATURES]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.ravel(), KEY_VARS):
    sns.boxplot(data=train_df.assign(classe=train_df["label"].map({0: "Normal", 1: "Attaque"})),
                x="classe", y=np.log1p(train_df[col]), ax=ax, showfliers=False)
    ax.set_title(f"log(1 + {col})"); ax.set_xlabel(""); ax.set_ylabel("")
plt.suptitle("Normal vs attaque sur quelques variables clés (train officiel)")
plt.tight_layout(); save_fig(fig, "eda_variables_cles"); plt.show()

**Ce que l'EDA implique pour le protocole**

- Le trafic normal est abondant : on peut apprendre le comportement normal sur un échantillon de `N_FIT` flux.
- Les familles rares (Worms, Shellcode…) ont peu d'exemples dans le test : leurs taux de détection auront une **forte incertitude** (à signaler dans les limites).
- Les doublons normaux sont nombreux : on les retire du jeu d'entraînement des détecteurs, sinon LOF calcule des distances nulles entre flux identiques et ses scores deviennent instables. Les doublons restent dans la validation et le test pour refléter le trafic réel.
- Les distributions sont très asymétriques (d'où le log dans le prétraitement), et certaines variables comme `sttl` séparent fortement les classes : ce point est testé en section 8.2.

✍️ *À compléter : 2–3 observations chiffrées tirées de vos graphiques (proportion d'attaques, familles les plus rares dans le test, variables les plus discriminantes visuellement).*

## 3. Protocole expérimental

### 3.1 Découpage des données

| Jeu | Contenu | Usage |
|---|---|---|
| **Pool d'entraînement normal** | Flux **normaux** de 80 % du train officiel, sans doublons | Entraînement des détecteurs (échantillon de `N_FIT` flux) |
| **Pool d'attaques d'entraînement** | Attaques des mêmes 80 % | Uniquement pour **contaminer** l'entraînement (Exp. 2) et entraîner le classifieur supervisé (Exp. 3) |
| **Validation** | 20 % restants du train officiel (stratifiés par famille) | Fixer les seuils (sur ses flux normaux), choisir le modèle à analyser |
| **Test** | Fichier test officiel | **Uniquement** pour mesurer les résultats finaux |

**Utilisation du test.** Aucune décision (seuil, hyperparamètre, choix du modèle) n'est prise sur le test. Seule exception, signalée : l'analyse des erreurs (section 5.6) est faite sur le test, car elle sert à **comprendre** les résultats et non à les améliorer.

### 3.2 Règle de seuil (commune à tous les modèles)

Un détecteur d'anomalies produit un **score** : plus il est élevé, plus le flux est jugé anormal. Il faut un seuil pour transformer ce score en alerte.

On fixe ce seuil **sans utiliser d'attaques** : c'est le quantile `1 − TARGET_FPR` des scores des flux **normaux de validation**. Avec `TARGET_FPR = 5 %`, chaque modèle laisse passer 95 % du trafic normal de validation sans alerte.

Pourquoi ce choix :
- c'est cohérent avec la question posée (on ne dispose pas d'exemples d'attaques pour régler le système) ;
- c'est réaliste pour un SOC, qui raisonne en **volume d'alertes acceptable** ;
- tous les modèles sont comparés **au même budget de faux positifs**, donc équitablement.

### 3.3 Métriques

L'attaque est la classe positive. L'accuracy n'est **jamais** utilisée seule, car elle est trompeuse avec des classes déséquilibrées.

| Métrique | Question à laquelle elle répond |
|---|---|
| **Rappel** | Quelle part des attaques est détectée ? |
| **Précision** | Parmi les alertes, quelle part sont de vraies attaques ? |
| **F1** et **Macro-F1** | Compromis précision/rappel ; Macro-F1 donne le même poids aux classes normale et attaque |
| **FPR** | Quelle part du trafic normal déclenche une fausse alerte ? |
| **PR-AUC** | Qualité du classement des scores, indépendamment du seuil (adaptée au déséquilibre) |
| **ROC-AUC** | Capacité globale à séparer normal et attaque, indépendamment du seuil |

### 3.4 Conditions comparables entre modèles

Mêmes variables, même prétraitement, **même échantillon d'entraînement**, même règle de seuil, mêmes graines. Les hyperparamètres sont fixés **a priori** (valeurs usuelles de la littérature, justifiées en section 4.2) et ne sont pas optimisés sur le test.

### 3.5 Pourquoi ce protocole ?

**Pourquoi trois jeux (entraînement, validation, test) et pas deux ?**
Chaque fois qu'on règle quelque chose en regardant des résultats (un seuil, un choix de modèle), ces résultats deviennent optimistes. Il faut donc un jeu pour **régler** (validation) et un autre, jamais regardé avant la fin, pour **mesurer** (test). Si l'on réglait le seuil sur le test, le score final mesurerait en partie notre ajustement plutôt que la vraie qualité du modèle.

**Pourquoi la validation vient du train officiel et pas du test ?**
Le test officiel doit rester intact. Prendre la validation dans le train garantit qu'aucune information du test n'intervient dans les décisions.

**Pourquoi stratifier la validation par famille (`attack_cat`) et pas seulement par `label` ?**
Une stratification par `label` garde la proportion normal/attaque, mais pourrait laisser presque aucun *Worms* dans la validation. En stratifiant par famille, chaque famille est représentée dans la même proportion qu'au départ.

**Pourquoi entraîner sur un échantillon de flux normaux et non sur tout le train ?**
- Le scénario étudié suppose qu'on ne dispose **pas** d'exemples d'attaques : les détecteurs ne voient que du normal.
- `N_FIT` est identique pour tous les modèles, condition d'une comparaison équitable (exigée par le sujet).

**Pourquoi le seuil est-il fixé uniquement avec des flux normaux ?**
C'est la seule règle cohérente avec la question posée. Si l'on utilisait des attaques étiquetées pour choisir le seuil, on supposerait connaître les attaques, ce que l'approche cherche justement à éviter. Cette règle a aussi un sens opérationnel : un SOC fixe d'abord **combien de fausses alertes il peut traiter**, puis mesure ce que ce budget permet de détecter.

**Pourquoi l'analyse des erreurs utilise le test ?**
Il faut des erreurs réelles et en nombre suffisant pour les étudier. Cette analyse ne modifie rien : aucun modèle ni seuil n'est changé après coup. C'est le seul usage du test hors mesure finale, et il est signalé (exigence du sujet).

### 3.6 Rappel des formules

Avec TP = attaques détectées, FN = attaques manquées, FP = flux normaux signalés à tort, TN = flux normaux non signalés :

| Métrique | Formule | Lecture |
|---|---|---|
| Rappel (taux de détection) | TP / (TP + FN) | 80 % = 8 attaques sur 10 détectées |
| Précision | TP / (TP + FP) | 60 % = 4 alertes sur 10 sont fausses |
| F1 | 2 × Précision × Rappel / (Précision + Rappel) | Élevé seulement si les deux sont élevés |
| Macro-F1 | Moyenne du F1 de la classe normale et du F1 de la classe attaque | Pénalise un modèle bon sur une seule classe |
| FPR | FP / (FP + TN) | 1 % = 1 flux normal sur 100 déclenche une alerte |
| PR-AUC | Aire sous la courbe précision-rappel, pour tous les seuils possibles | Vaut la proportion d'attaques pour un modèle aléatoire, 1 pour un modèle parfait |
| ROC-AUC | Probabilité qu'une attaque tirée au hasard ait un score plus élevé qu'un flux normal tiré au hasard | 0,5 = hasard, 1 = parfait |

**Pourquoi PR-AUC plutôt que ROC-AUC comme critère principal ?** La ROC-AUC intègre le FPR, dont le dénominateur est le grand nombre de flux normaux : quelques milliers de fausses alertes y paraissent négligeables. La PR-AUC mesure directement la qualité des alertes (précision), ce qui compte pour un analyste.

In [ ]:
train_part, val_df = train_test_split(
    train_df, test_size=VAL_SIZE, stratify=train_df["attack_cat"], random_state=RANDOM_STATE)

normal_pool = train_part[train_part["label"] == 0].drop_duplicates(subset=FEATURES)
attack_pool = train_part[train_part["label"] == 1]

X_val, y_val = val_df[FEATURES], val_df["label"].to_numpy()
X_test, y_test = test_df[FEATURES], test_df["label"].to_numpy()
val_normal = y_val == 0
X_val_normal = X_val[val_normal]

N_FIT = min(N_FIT, len(normal_pool))
X_fit = normal_pool.sample(n=N_FIT, random_state=RANDOM_STATE)[FEATURES]

num_cols = [c for c in FEATURES if c not in CAT_COLS]
CONT_COLS = [c for c in num_cols if normal_pool[c].nunique() > 50]
DISC_COLS = [c for c in num_cols if c not in CONT_COLS]

splits = pd.DataFrame({
    "flux": [len(normal_pool), len(attack_pool), N_FIT, len(val_df), len(test_df)],
    "dont attaques": [0, len(attack_pool), 0, int(y_val.sum()), int(y_test.sum())],
}, index=["Pool normal (sans doublons)", "Pool attaques", "Entraînement détecteurs (N_FIT)",
          "Validation", "Test officiel"])
save_table(splits, "decoupage")
splits

## 4. Prétraitement, baseline et détecteurs

### 4.1 Pipeline de prétraitement

Le prétraitement est intégré dans un `Pipeline` scikit-learn avec chaque modèle. Toutes ses statistiques (percentiles, moyennes, écarts-types, catégories gardées) sont donc apprises **sur le seul jeu d'entraînement** du modèle, puis appliquées telles quelles à la validation et au test. C'est ce qui évite toute fuite de données (pas de scaling ni de sélection sur l'ensemble complet).

| Variables | Transformation | Raison |
|---|---|---|
| Continues (> 50 valeurs distinctes) | Écrêtage au 99e percentile → `log1p` → standardisation | Distributions à très longue queue ; les détecteurs à base de distance (LOF, OC-SVM) y sont très sensibles |
| Discrètes (compteurs `ct_*`, TTL, indicateurs) | Standardisation | Même échelle pour toutes les variables |
| Catégorielles (`proto`, `service`, `state`) | One-hot des 5 modalités les plus fréquentes + « rare » | Évite l'explosion du nombre de colonnes ; une modalité jamais vue va dans « rare » |

Dans les expériences 1 et 3, le prétraitement des détecteurs est appris **uniquement sur du trafic normal** : une valeur ou un protocole inhabituel pour le trafic normal ressort alors naturellement comme anormal.

#### Pourquoi ces transformations, dans cet ordre ?

1. **Écrêtage d'abord** : on plafonne les valeurs démesurées avant tout calcul, pour qu'elles ne faussent ni le log ni la moyenne et l'écart-type du `StandardScaler`. Le seuil (99e percentile) est **appris sur l'entraînement** : un flux de test plus grand que tout ce qui a été vu est ramené à cette valeur maximale, il reste donc parmi les plus extrêmes.
2. **Log ensuite** : sur des variables comme `sbytes` (de quelques dizaines d'octets à plusieurs millions), une différence de 100 octets n'a pas le même sens à 200 qu'à 2 000 000. Le log compare des **rapports** plutôt que des différences, ce qui correspond mieux à la façon dont ces grandeurs varient.
3. **Standardisation enfin** : chaque variable a alors une moyenne de 0 et un écart-type de 1 **sur le trafic normal**. Une valeur à +3 signifie « 3 écarts-types au-dessus du normal », ce qui donne un sens direct aux distances calculées par LOF, One-Class SVM et la baseline.

**Pourquoi distinguer variables continues et discrètes ?** Les compteurs (`ct_*`), indicateurs (`is_*`) et TTL prennent peu de valeurs entières : les écrêter ou leur appliquer un log n'apporte rien et peut écraser des valeurs significatives (par exemple un TTL de 254). Le seuil de 50 valeurs distinctes, repris du notebook d'origine, sert seulement à séparer ces deux types de variables.

**Pourquoi le même prétraitement pour Isolation Forest, qui n'en a pas besoin ?** Isolation Forest est insensible aux transformations monotones (log, mise à l'échelle) car il coupe sur des seuils. Garder le même prétraitement pour tous assure que seules les **méthodes** diffèrent entre les modèles, pas les données qu'elles reçoivent.

**Pourquoi un `Pipeline` ?** Il garantit mécaniquement que `fit` n'est appelé que sur les données d'entraînement, et que validation et test passent par exactement les mêmes transformations. C'est plus sûr qu'une série d'étapes manuelles, où une erreur de variable suffit à créer une fuite.

In [ ]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Plafonne chaque colonne à un quantile appris sur les données d'entraînement."""
    def __init__(self, upper_quantile=0.99):
        self.upper_quantile = upper_quantile
    def fit(self, X, y=None):
        self.upper_ = np.quantile(np.asarray(X, dtype=float), self.upper_quantile, axis=0)
        return self
    def transform(self, X):
        return np.minimum(np.asarray(X, dtype=float), self.upper_)
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def log1p_pos(X):
    return np.log1p(np.clip(X, 0, None))


def build_preprocessor(cont_cols, disc_cols, cat_cols):
    cont_pipe = Pipeline([
        ("clip", QuantileClipper(CLIP_QUANTILE)),
        ("log", FunctionTransformer(log1p_pos, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])
    return ColumnTransformer([
        ("cont", cont_pipe, cont_cols),
        ("disc", StandardScaler(), disc_cols),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", max_categories=6), cat_cols),
    ], sparse_threshold=0)


def make_pipeline(model, cols=None):
    cont, disc, cat = cols or (CONT_COLS, DISC_COLS, CAT_COLS)
    return Pipeline([("prep", build_preprocessor(cont, disc, cat)), ("model", model)])

### 4.2 Baseline et détecteurs d'anomalies

Le sujet demande une **baseline** avant les modèles plus complexes. On prend la plus simple possible :

- **Baseline z-score** : on mesure, pour chaque flux, l'écart moyen (en nombre d'écarts-types) de ses variables par rapport au trafic normal. Pas d'apprentissage au-delà d'une moyenne et d'un écart-type. Si les méthodes avancées ne font pas mieux, elles n'apportent rien.

Puis les trois détecteurs imposés :

| Modèle | Principe | Hyperparamètres (fixés a priori) |
|---|---|---|
| **Isolation Forest** | Construit des arbres aléatoires ; un point **facile à isoler** (peu de coupures) est anormal | 200 arbres, `max_samples=256` (valeurs recommandées par les auteurs de la méthode) |
| **Local Outlier Factor** | Compare la **densité locale** d'un point à celle de ses voisins ; un point dans une zone moins dense que son voisinage est anormal | `n_neighbors=35` (plage usuelle 20–50), `novelty=True` pour scorer de nouveaux flux |
| **One-Class SVM** | Apprend une **frontière** autour du trafic normal dans un espace transformé (noyau RBF) | `nu=0.05` (cohérent avec un FPR visé de 5 %), `gamma="scale"` |

Pour tous les modèles, le score d'anomalie est `−score_samples` : **plus il est grand, plus le flux est anormal**. Le paramètre `contamination` d'Isolation Forest n'est pas utilisé, car le seuil est fixé par la règle commune de la section 3.2.

#### Comment fonctionne chaque détecteur, et pourquoi ces hyperparamètres ?

**Baseline z-score**
- *Principe* : pour chaque variable, on calcule à combien d'écarts-types du trafic normal se trouve le flux, puis on fait la moyenne des valeurs absolues. Plus le score est élevé, plus le flux s'écarte du normal sur l'ensemble de ses variables.
- *Pourquoi* : aucun hyperparamètre, résultat explicable en une phrase. Elle traite chaque variable séparément et ignore les combinaisons de variables : si les modèles avancés font mieux, c'est qu'ils exploitent ces combinaisons.

**Isolation Forest** (Liu et al., 2008)
- *Principe* : on construit des arbres en choisissant au hasard une variable et un seuil de coupure, jusqu'à isoler chaque point. Un point anormal, éloigné des autres, est isolé en **peu de coupures**. Le score dépend de la profondeur moyenne à laquelle le flux est isolé.
- `n_estimators=200` : les auteurs montrent que les scores se stabilisent vers 100 arbres ; 200 réduit encore la variabilité pour un coût faible.
- `max_samples=256` : valeur recommandée par les auteurs. Chaque arbre est construit sur un petit sous-échantillon, ce qui rend les anomalies plus faciles à isoler (moins de points normaux autour pour les masquer) et l'entraînement très rapide.

**Local Outlier Factor** (Breunig et al., 2000)
- *Principe* : on compare la densité autour d'un flux (distance à ses voisins) à la densité autour de ses voisins. Un score proche de 1 signifie « aussi dense que son voisinage » ; un score nettement supérieur à 1 signifie « isolé par rapport à ses voisins ». LOF détecte ainsi des anomalies **locales**, même dans une zone de valeurs banales.
- `n_neighbors=35` : trop peu de voisins rend la densité instable (bruit) ; trop de voisins mélange des groupes de flux différents. La documentation scikit-learn recommande 20 comme valeur générale ; on prend un peu plus car le trafic contient de nombreux flux très semblables (groupes denses), qu'il faut dépasser pour estimer une densité locale fiable. La plage 20–50 est usuelle.
- `novelty=True` : obligatoire pour scorer des flux **nouveaux** (validation, test) avec un modèle appris sur l'entraînement. Sans ce paramètre, LOF ne score que les points d'entraînement.

**One-Class SVM** (Schölkopf et al., 2001)
- *Principe* : on cherche une frontière qui entoure la majeure partie du trafic normal dans un espace transformé par un noyau. Le noyau RBF permet une frontière courbe, adaptée à un normal composé de plusieurs groupes (web, DNS, etc.). Le score est la distance signée à cette frontière.
- `nu=0.05` : `nu` est une borne supérieure de la part de points d'entraînement laissés hors de la frontière. 5 % est cohérent avec le budget de fausses alertes visé (`TARGET_FPR`).
- `gamma="scale"` : règle automatique de scikit-learn, `1 / (nombre de variables × variance des données)`. Elle adapte la largeur du noyau aux données standardisées sans réglage manuel. Un `gamma` trop grand donnerait une frontière collée à chaque point (sur-apprentissage), trop petit une frontière trop lisse.

**Pourquoi ne pas optimiser ces hyperparamètres ?** Il faudrait un critère, donc des attaques étiquetées, ce qui contredit le scénario. Le seul réglage autorisé ici serait non supervisé, et il n'existe pas de critère standard pour cela. On fixe donc des valeurs reconnues, identiques à chaque expérience, et on signale cette limite.

In [ ]:
class ZScoreBaseline(BaseEstimator):
    """Baseline : écart absolu moyen (en écarts-types) au trafic normal d'entraînement."""
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.mu_, self.sd_ = X.mean(axis=0), X.std(axis=0)
        self.sd_[self.sd_ == 0] = 1.0
        return self
    def score_samples(self, X):
        # Convention scikit-learn : score élevé = normal. On renvoie donc l'opposé de l'écart moyen.
        X = np.asarray(X, dtype=float)
        return -np.abs((X - self.mu_) / self.sd_).mean(axis=1)


def make_detectors(seed=RANDOM_STATE):
    return {
        "Baseline z-score": ZScoreBaseline(),
        "Isolation Forest": IsolationForest(n_estimators=200, max_samples=256,
                                            random_state=seed, n_jobs=-1),
        "LOF": LocalOutlierFactor(n_neighbors=35, novelty=True, n_jobs=-1),
        "One-Class SVM": OneClassSVM(kernel="rbf", gamma="scale", nu=0.05),
    }

DETECTORS = list(make_detectors())


def anomaly_score(pipe, X):
    """Plus le score est élevé, plus le flux est anormal."""
    return -pipe.score_samples(X)


def threshold_from_normals(scores_normal, target_fpr=TARGET_FPR):
    """Seuil tel qu'une part target_fpr des flux normaux le dépasse.
    Exemple : target_fpr = 5 % -> 95e percentile des scores normaux de validation.
    Aucune attaque n'est utilisée (cohérent avec le scénario étudié)."""
    return np.quantile(scores_normal, 1 - target_fpr)


def detection_metrics(y, scores, thr):
    pred = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "Précision": precision_score(y, pred, zero_division=0),
        "Rappel": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "Macro-F1": f1_score(y, pred, average="macro", zero_division=0),
        "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
        "ROC-AUC": roc_auc_score(y, scores),
        "PR-AUC": average_precision_score(y, scores),
        "FP": int(fp), "FN": int(fn),
    }


PCT_COLS = ["Précision", "Rappel", "F1", "Macro-F1", "FPR", "ROC-AUC", "PR-AUC"]

def fmt(df):
    cols = [c for c in PCT_COLS if c in df.columns]
    return df.style.format({c: "{:.1%}" for c in cols}, precision=2)

## 5. Expérience 1 — Comparaison de détecteurs d'anomalies

| | |
|---|---|
| **Question de recherche** | Les détecteurs d'anomalies appris sur du trafic normal détectent-ils les attaques, et avec quels compromis entre détection et fausses alertes ? |
| **Hypothèse H1** | Les méthodes auront des compromis différents entre détection et faux positifs. |
| **Protocole** | Les 4 modèles sont entraînés sur le **même** échantillon de `N_FIT` flux normaux, avec le même prétraitement. Seuil fixé par la règle commune (FPR visé sur les normaux de validation). Résultats mesurés sur le test officiel. Le modèle analysé en détail est choisi sur la **PR-AUC de validation**. |
| **Résultats attendus** | Rappel, précision, F1, FPR, distribution des scores d'anomalie. |


#### Pourquoi ce protocole ?
- **Même échantillon `X_fit` pour tous** : si chaque modèle avait son propre tirage, une différence de résultats pourrait venir du tirage et non de la méthode.
- **Évaluation à seuil fixé ET sans seuil** : les métriques au seuil (rappel, FPR…) décrivent ce que verrait un SOC ; la PR-AUC et la ROC-AUC décrivent la capacité du modèle à classer, quel que soit le seuil. Les deux sont nécessaires : un modèle peut bien classer mais être pénalisé par un seuil mal adapté.
- **Choix du modèle analysé sur la validation** : le sujet interdit de choisir le meilleur modèle sur le test. On retient le meilleur des trois détecteurs (hors baseline) selon la PR-AUC de validation.

### 5.1 Entraînement et évaluation

In [ ]:
exp1_pipes, exp1_thr, val_scores, test_scores = {}, {}, {}, {}
rows_val, rows_test = {}, {}

for name, est in make_detectors().items():
    pipe = make_pipeline(est)
    t0 = time.perf_counter(); pipe.fit(X_fit); t_fit = time.perf_counter() - t0

    t0 = time.perf_counter()
    s_val, s_test = anomaly_score(pipe, X_val), anomaly_score(pipe, X_test)
    t_score = time.perf_counter() - t0

    thr = threshold_from_normals(s_val[val_normal])
    exp1_pipes[name], exp1_thr[name] = pipe, thr
    val_scores[name], test_scores[name] = s_val, s_test
    rows_val[name] = detection_metrics(y_val, s_val, thr)
    rows_test[name] = detection_metrics(y_test, s_test, thr) | {
        "Entraînement (s)": t_fit, "Scoring val+test (s)": t_score}
    print(f"{name:18s} entraîné en {t_fit:6.1f} s | PR-AUC test = {rows_test[name]['PR-AUC']:.3f}")

exp1_val = pd.DataFrame(rows_val).T
exp1_test = pd.DataFrame(rows_test).T
save_table(exp1_val, "exp1_validation"); save_table(exp1_test, "exp1_test")

best_name = exp1_val.loc[DETECTORS[1:], "PR-AUC"].idxmax()
print(f"\nDétecteur retenu pour l'analyse détaillée (meilleure PR-AUC de VALIDATION) : {best_name}")

### 5.2 Résultats sur le test officiel (seuil à FPR visé = `TARGET_FPR` sur la validation)

In [ ]:
fmt(exp1_test)

### 5.3 Distribution des scores d'anomalie

Pour chaque modèle : histogramme des scores des flux normaux et des attaques du test, avec le seuil. Plus les deux distributions sont séparées, meilleur est le détecteur. Le chevauchement correspond aux attaques que le modèle ne peut pas distinguer du normal.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, name in zip(axes.ravel(), DETECTORS):
    s = test_scores[name]
    lo, hi = np.quantile(s, [0.001, 0.995])
    sc = np.clip(s, lo, hi)
    bins = np.linspace(lo, hi, 80)
    ax.hist(sc[y_test == 0], bins=bins, alpha=0.6, density=True, label="Normal")
    ax.hist(sc[y_test == 1], bins=bins, alpha=0.6, density=True, label="Attaque")
    ax.axvline(np.clip(exp1_thr[name], lo, hi), color="black", ls="--", label="Seuil")
    ax.set_title(name); ax.set_xlabel("Score d'anomalie (écrêté pour l'affichage)")
    ax.legend()
plt.suptitle("Expérience 1 — Distribution des scores d'anomalie sur le test")
plt.tight_layout(); save_fig(fig, "exp1_distributions_scores"); plt.show()

### 5.4 Compromis détection / faux positifs (H1)

On fait varier le budget de faux positifs (seuils fixés sur la validation à 1, 2, 5 et 10 % de FPR) et on mesure le rappel obtenu sur le test. Si les classements des modèles changent selon le budget, ou si leurs courbes ROC/PR se croisent, ils offrent des compromis différents.

**Pourquoi ces quatre budgets (1, 2, 5, 10 %) ?** Ils couvrent la plage réaliste pour un SOC, du budget strict (1 %) au budget large (10 %). Un modèle peut être le meilleur à 10 % de FPR et le moins bon à 1 % : c'est précisément ce qu'on appelle un compromis différent.

**Pourquoi seuils sur la validation, mesure sur le test ?** Comme partout ailleurs : le FPR obtenu sur le test peut légèrement s'écarter du FPR visé, et cet écart est lui-même une information (stabilité du seuil d'un jeu de données à l'autre).

**Critère de décision pour H1, fixé avant d'observer les résultats** : H1 est considérée comme soutenue si le classement des modèles change selon le budget de FPR, **ou** si l'écart de rappel entre modèles dépasse 5 points au seuil commun. 5 points est un choix de **pertinence pratique** : sur 10 000 attaques, c'est 500 attaques détectées en plus ou en moins. On ne fait pas de test statistique formel, car on dispose d'un seul jeu de test et de peu de modèles.

In [ ]:
rows = []
for name in DETECTORS:
    for f in FPR_LEVELS:
        thr = threshold_from_normals(val_scores[name][val_normal], f)
        m = detection_metrics(y_test, test_scores[name], thr)
        rows.append({"modèle": name, "FPR visé": f, "Rappel test": m["Rappel"], "FPR test": m["FPR"],
                     "Précision test": m["Précision"]})
tradeoff = pd.DataFrame(rows)
save_table(tradeoff, "exp1_compromis")
recall_at = tradeoff.pivot(index="modèle", columns="FPR visé", values="Rappel test")
fpr_at = tradeoff.pivot(index="modèle", columns="FPR visé", values="FPR test")
print("Rappel sur le test selon le FPR visé :")
display(recall_at.style.format("{:.1%}").background_gradient(cmap="Greens", axis=0))
print("FPR réellement obtenu sur le test :")
display(fpr_at.style.format("{:.1%}"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name in DETECTORS:
    RocCurveDisplay.from_predictions(y_test, test_scores[name], name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, test_scores[name], name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_title("Courbes ROC (test)"); axes[1].set_title("Courbes précision-rappel (test)")
plt.tight_layout(); save_fig(fig, "exp1_roc_pr"); plt.show()

In [ ]:
# Observation automatique pour H1
ranking = recall_at.rank(axis=0, ascending=False)
classement_change = (ranking.nunique(axis=1) > 1).any()
ecart_rappel = exp1_test["Rappel"].max() - exp1_test["Rappel"].min()
ecart_fpr = exp1_test["FPR"].max() - exp1_test["FPR"].min()

print("OBSERVATION (H1)")
print(f"- Au seuil commun, le rappel varie de {exp1_test['Rappel'].min():.1%} à {exp1_test['Rappel'].max():.1%} "
      f"(écart {ecart_rappel:.1%}) et le FPR test de {exp1_test['FPR'].min():.1%} à {exp1_test['FPR'].max():.1%}.")
print(f"- Le classement des modèles {'CHANGE' if classement_change else 'ne change pas'} selon le budget de faux positifs.")
for f in FPR_LEVELS:
    print(f"  · FPR visé {f:.0%} : meilleur rappel = {recall_at[f].idxmax()} ({recall_at[f].max():.1%})")
verdict_h1 = classement_change or ecart_rappel > 0.05
print(f"-> H1 {'SOUTENUE' if verdict_h1 else 'NON SOUTENUE'} par ces mesures "
      f"(critère : classement qui change ou écart de rappel > 5 points).")

### 5.5 Détection par famille d'attaque

Taux de détection de chaque famille au seuil commun (pour la ligne *Normal*, c'est le FPR). Cette table sert à répondre à **Q2** : quelles attaques ressemblent trop au trafic normal ?

In [ ]:
def family_detection(scores, thr, frame=test_df):
    pred = scores >= thr
    return pd.Series(pred, index=frame.index).groupby(frame["attack_cat"]).mean()

fam_exp1 = pd.DataFrame({name: family_detection(test_scores[name], exp1_thr[name]) for name in DETECTORS})
fam_exp1["n (test)"] = test_df["attack_cat"].value_counts()
fam_exp1 = fam_exp1.sort_values(best_name, ascending=False)
save_table(fam_exp1, "exp1_detection_par_famille")
attack_rows = [f for f in fam_exp1.index if f != "Normal"]  # la ligne Normal (FPR) n'est pas colorée : bas = bien
(fam_exp1.style.format({c: "{:.1%}" for c in DETECTORS})
 .background_gradient(subset=pd.IndexSlice[attack_rows, DETECTORS], cmap="RdYlGn", vmin=0, vmax=1))

### 5.6 Analyse des faux positifs et faux négatifs (modèle retenu)

> **Usage du test signalé** : cette analyse est faite sur le test pour comprendre les erreurs ; elle ne modifie aucun modèle ni aucun seuil.

- **Faux positifs** : quels flux normaux déclenchent des alertes ? On regarde leur répartition par service et par état de connexion.
- **Faux négatifs** : quelles attaques passent ? (table par famille ci-dessus)
- **Profil** : on compare la médiane de quelques variables entre vrais négatifs, faux positifs, vrais positifs et faux négatifs.

In [ ]:
pred_best = test_scores[best_name] >= exp1_thr[best_name]
is_normal = y_test == 0

for col in ["service", "state"]:
    t = (pd.DataFrame({col: test_df[col].to_numpy(), "alerte": pred_best})[is_normal]
         .groupby(col)["alerte"].agg(flux_normaux="count", taux_FP="mean"))
    t["part des FP"] = t["taux_FP"] * t["flux_normaux"] / max(1, pred_best[is_normal].sum())
    t = t.sort_values("part des FP", ascending=False).head(8)
    save_table(t, f"exp1_fp_par_{col}")
    print(f"Faux positifs du modèle {best_name} par {col} :")
    display(t.style.format({"taux_FP": "{:.1%}", "part des FP": "{:.1%}"}))

outcome = np.select(
    [is_normal & ~pred_best, is_normal & pred_best, ~is_normal & pred_best, ~is_normal & ~pred_best],
    ["Vrai négatif", "Faux positif", "Vrai positif", "Faux négatif"], default="")
PROFILE_VARS = [c for c in ["dur", "sbytes", "dbytes", "spkts", "dpkts", "rate", "sttl", "dttl",
                            "smean", "dmean", "ct_srv_src", "ct_dst_src_ltm"] if c in FEATURES]
profile = test_df[PROFILE_VARS].groupby(outcome).median().T
profile = profile[[c for c in ["Vrai négatif", "Faux positif", "Vrai positif", "Faux négatif"] if c in profile]]
save_table(profile, "exp1_profil_erreurs")
print("Médianes des variables selon le type de décision :")
profile

### 5.7 Analyse de l'expérience 1

✍️ *À rédiger avec vos chiffres. Trame proposée :*

**Observation expérimentale** — Reprendre la sortie « OBSERVATION (H1) » : écart de rappel et de FPR entre modèles, modèle le plus performant à chaque budget de FPR, PR-AUC de chacun, position de la baseline. Citer les familles les mieux et les moins bien détectées.

**Interprétation ML** —
- *Isolation Forest* isole bien les points aux valeurs extrêmes sur quelques variables (gros volumes, débits atypiques), mais repère moins bien une attaque « au milieu » de la distribution normale.
- *LOF* raisonne localement : il détecte un flux isolé même dans une zone de valeurs banales, mais il est sensible à l'échelle des variables et aux zones normales peu denses (source de faux positifs).
- *One-Class SVM* trace une frontière globale lisse : sa qualité dépend fortement de `gamma` et `nu`.
- Si la baseline z-score fait presque aussi bien, cela signifie que la séparation repose surtout sur des écarts simples sur quelques variables.
- Confronter ces explications aux distributions de scores (5.3) : un fort chevauchement explique un rappel bas.

**Interprétation cybersécurité** — Quels flux normaux génèrent des alertes (5.6) : services rares ou connexions inhabituelles mais légitimes ? Quelles familles passent sous le radar (5.5) ? Un modèle à haut rappel mais FPR élevé saturerait les analystes ; un modèle à FPR très bas manquerait des attaques.

**Limites** — Hyperparamètres non optimisés ; un seul échantillon d'entraînement de `N_FIT` flux ; le seuil dépend de la validation ; les familles rares ont peu d'exemples (taux instables) ; dataset simulé en laboratoire.

## 6. Expérience 2 — Contamination du jeu d'entraînement

| | |
|---|---|
| **Question de recherche** | Les détecteurs restent-ils fiables si le trafic supposé normal contient en réalité des attaques non repérées ? |
| **Hypothèse H2** | L'augmentation de la contamination réduira la capacité à distinguer normal et attaque. |
| **Protocole** | Pour chaque niveau de contamination *c* ∈ {0, 1, 2, 5, 10 %}, on construit un jeu d'entraînement de `N_FIT` flux dont une part *c* sont des attaques tirées au hasard dans le pool d'attaques d'entraînement (toutes familles). Le prétraitement est aussi appris sur ces données contaminées, comme en situation réelle. On réentraîne les 4 modèles. Le seuil suit la même règle (FPR visé sur les normaux propres de validation), ce qui isole l'effet de la contamination sur le **modèle**. L'expérience est répétée `N_SEEDS` fois avec des tirages différents pour estimer la variabilité. |
| **Résultats attendus** | Courbes contamination → rappel, FPR, F1 (et PR-AUC, qui mesure la séparabilité sans dépendre du seuil). |

**Pourquoi c'est réaliste** : dans un SOC, le trafic « normal » de référence n'est jamais garanti propre ; des attaques non détectées peuvent s'y cacher.

#### Pourquoi ces choix ?

- **Taille d'entraînement constante (`N_FIT`)** : on **remplace** des flux normaux par des attaques au lieu d'en ajouter. Sinon, le jeu grossirait avec la contamination, et on ne saurait plus si l'effet vient de la contamination ou de la quantité de données.
- **Attaques tirées au hasard dans toutes les familles** : c'est l'hypothèse la plus neutre, qui reproduit la répartition des attaques du dataset. Une contamination par une seule famille serait un autre scénario, plus proche d'un attaquant persistant ; il est cité en limite.
- **Prétraitement appris sur les données contaminées** : en situation réelle, on ne sait pas que les données sont contaminées ; les moyennes et écarts-types seraient donc eux aussi calculés avec les attaques. Le protocole reproduit ce cas.
- **Seuil fixé sur des normaux propres** : on veut mesurer l'effet de la contamination sur le **modèle**. Si la validation était aussi contaminée, le seuil changerait en même temps et on ne pourrait pas séparer les deux effets.
- **Plusieurs répétitions** : à 1 % de contamination, on n'injecte que 200 attaques ; selon celles qui sont tirées, le résultat varie. Les répétitions mesurent cette variabilité (barres d'erreur) et évitent d'interpréter un effet du hasard comme une tendance.
- **Critère de décision pour H2, fixé à l'avance** : H2 est soutenue pour un modèle si sa PR-AUC moyenne est plus basse à 10 % qu'à 0 %. La PR-AUC est choisie car elle mesure la **capacité à distinguer** normal et attaque sans dépendre du seuil, ce que formule exactement H2.
- **Corrélation de Spearman** : elle mesure si la PR-AUC baisse de façon **régulière** quand la contamination augmente (ρ proche de −1), sans supposer une baisse linéaire. Avec 5 niveaux et peu de répétitions, la p-valeur est indicative seulement.

In [ ]:
exp2_rows = []
t_start = time.perf_counter()
for seed in range(N_SEEDS):
    rs = RANDOM_STATE + seed
    for c in CONTAMINATION_LEVELS:
        n_att = int(round(c * N_FIT))
        normals = normal_pool.sample(n=N_FIT - n_att, random_state=rs)
        attacks = attack_pool.sample(n=n_att, random_state=rs) if n_att else attack_pool.iloc[:0]
        X_c = pd.concat([normals, attacks])[FEATURES].sample(frac=1, random_state=rs)
        for name, est in make_detectors(rs).items():
            pipe = make_pipeline(est).fit(X_c)
            thr = threshold_from_normals(anomaly_score(pipe, X_val_normal))
            m = detection_metrics(y_test, anomaly_score(pipe, X_test), thr)
            exp2_rows.append({"répétition": seed, "contamination": c, "modèle": name, **m})
        print(f"répétition {seed + 1}/{N_SEEDS} | contamination {c:>4.0%} terminée "
              f"({time.perf_counter() - t_start:.0f} s écoulées)")

exp2 = pd.DataFrame(exp2_rows)
save_table(exp2, "exp2_resultats_bruts")
exp2_summary = exp2.groupby(["modèle", "contamination"])[["Rappel", "FPR", "F1", "PR-AUC", "ROC-AUC"]].agg(["mean", "std"])
save_table(exp2_summary, "exp2_resume")
exp2_summary.style.format("{:.1%}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric in zip(axes.ravel(), ["Rappel", "FPR", "F1", "PR-AUC"]):
    for name in DETECTORS:
        g = exp2[exp2["modèle"] == name].groupby("contamination")[metric]
        mean, std = g.mean(), g.std().fillna(0)
        ax.errorbar(mean.index * 100, mean.values, yerr=std.values, marker="o", capsize=3, label=name)
    ax.set_title(metric); ax.set_xlabel("Contamination de l'entraînement (%)")
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
axes[0, 0].legend()
plt.suptitle(f"Expérience 2 — Effet de la contamination (moyenne ± écart-type sur {N_SEEDS} répétition(s))")
plt.tight_layout(); save_fig(fig, "exp2_courbes_contamination"); plt.show()

In [ ]:
# Observation automatique pour H2
rows = []
c0, cmax = min(CONTAMINATION_LEVELS), max(CONTAMINATION_LEVELS)
for name in DETECTORS:
    sub = exp2[exp2["modèle"] == name]
    m = sub.groupby("contamination")[["PR-AUC", "ROC-AUC", "Rappel", "FPR"]].mean()
    rho, p = spearmanr(sub["contamination"], sub["PR-AUC"])
    rows.append({"modèle": name,
                 f"PR-AUC à {c0:.0%}": m.loc[c0, "PR-AUC"], f"PR-AUC à {cmax:.0%}": m.loc[cmax, "PR-AUC"],
                 "Δ PR-AUC": m.loc[cmax, "PR-AUC"] - m.loc[c0, "PR-AUC"],
                 "Δ ROC-AUC": m.loc[cmax, "ROC-AUC"] - m.loc[c0, "ROC-AUC"],
                 "Δ Rappel": m.loc[cmax, "Rappel"] - m.loc[c0, "Rappel"],
                 "Spearman ρ (contam., PR-AUC)": rho, "p-valeur": p})
h2 = pd.DataFrame(rows).set_index("modèle")
save_table(h2, "exp2_verdict_h2")
display(h2.style.format("{:+.3f}", subset=["Δ PR-AUC", "Δ ROC-AUC", "Δ Rappel"])
          .format("{:.1%}", subset=[f"PR-AUC à {c0:.0%}", f"PR-AUC à {cmax:.0%}"])
          .format("{:.2f}", subset=["Spearman ρ (contam., PR-AUC)"]).format("{:.3f}", subset=["p-valeur"]))

print("OBSERVATION (H2)")
for name, r in h2.iterrows():
    sens = "baisse" if r["Δ PR-AUC"] < 0 else "hausse"
    print(f"- {name:18s} : PR-AUC en {sens} de {abs(r['Δ PR-AUC']):.3f} entre {c0:.0%} et {cmax:.0%} "
          f"(ρ = {r['Spearman ρ (contam., PR-AUC)']:.2f}).")
n_deg = int((h2["Δ PR-AUC"] < 0).sum())
print(f"-> H2 soutenue pour {n_deg}/{len(h2)} modèles (critère : PR-AUC plus faible à {cmax:.0%} qu'à {c0:.0%}). "
      "Une corrélation n'est pas une preuve causale à elle seule, mais ici la contamination est la seule variable "
      "modifiée par le protocole.")

### 6.1 Analyse de l'expérience 2

✍️ *À rédiger avec vos chiffres. Trame proposée :*

**Observation expérimentale** — Évolution du rappel, du FPR, du F1 et de la PR-AUC entre 0 et 10 % (table ci-dessus), modèle le plus robuste (plus faible Δ), dispersion entre répétitions.

**Interprétation ML** — Les attaques injectées deviennent « normales » pour le modèle : il élargit sa notion du normal et accepte des flux qui leur ressemblent.
- *LOF* et *One-Class SVM* sont souvent les plus touchés : quelques attaques similaires suffisent à créer une zone dense (LOF) ou à déplacer la frontière (OC-SVM).
- *Isolation Forest* peut être plus robuste, car il travaille sur des sous-échantillons de 256 flux, mais une contamination concentrée sur une famille rend aussi ces attaques faciles à « mal isoler ».
- Si le FPR reste stable alors que le rappel baisse, c'est l'effet attendu de la règle de seuil fixée sur des normaux propres.

**Interprétation cybersécurité** — Un détecteur appris sur une période déjà compromise risque de considérer l'attaquant comme normal (ce qui est le scénario d'une compromission persistante). D'où l'importance de nettoyer les données de référence, de réentraîner régulièrement, et de combiner avec d'autres sources de détection.

**Limites** — Contamination tirée au hasard sur toutes les familles (une vraie contamination serait souvent concentrée sur un seul attaquant) ; seuil fixé sur une validation propre (en réalité, elle serait aussi contaminée, ce qui dégraderait davantage) ; nombre de répétitions limité.

## 7. Expérience 3 — Simulation zero-day

| | |
|---|---|
| **Question de recherche** | Face à des familles d'attaques **totalement absentes de l'apprentissage**, un détecteur d'anomalies fait-il mieux qu'un classifieur supervisé entraîné sur les autres familles ? |
| **Hypothèse H3** | Certaines méthodes d'anomalies seront moins précises globalement mais plus utiles sur certaines familles inconnues. |
| **Protocole** | Familles « inconnues » : `UNKNOWN_FAMILIES`. Elles sont retirées **du train et de la validation**, et n'apparaissent que dans le test. Détecteurs d'anomalies : ceux de l'expérience 1, entraînés sur du trafic normal seulement (ils n'ont vu aucune famille d'attaque). Classifieur supervisé : Random Forest entraîné sur le normal + les familles **connues**. Seuil : même règle pour tous (FPR visé sur les normaux de validation), afin de comparer au même budget de fausses alertes. |
| **Résultats attendus** | Comparaison connues / inconnues et analyse par famille. |

**Choix des familles inconnues** : elles couvrent des comportements variés (balayage réseau pour *Reconnaissance*, porte dérobée pour *Backdoor*, charge malveillante pour *Shellcode*, propagation pour *Worms*). On ne qualifie une famille d'« inconnue » que parce qu'**aucune** de ses observations n'a servi à l'entraînement ni au réglage.

> **Note** : le choix du détecteur analysé en 5.6 utilisait une validation contenant ces familles. Ici, **les quatre détecteurs sont présentés** sans sélection, ce qui évite ce biais.

#### Pourquoi ces choix ?

**Pourquoi ces quatre familles ?**

| Famille | Comportement | Pourquoi elle est intéressante comme « inconnue » |
|---|---|---|
| Reconnaissance | Balayage de ports et d'hôtes | Famille de taille moyenne (plusieurs milliers de flux de test) : taux de détection mesuré de façon fiable. Comportement typique de la phase préparatoire d'une attaque |
| Backdoor | Accès caché à une machine compromise | Trafic souvent discret, proche d'un usage normal : test difficile |
| Shellcode | Envoi de code malveillant dans une charge utile | Contenu malveillant mais flux parfois banal au niveau des statistiques réseau |
| Worms | Propagation automatique | Très rare : représente l'attaque nouvelle peu fréquente ; taux à interpréter avec prudence |

Les grandes familles (*Generic*, *Exploits*, *Fuzzers*, *DoS*) restent **connues** : le classifieur supervisé garde ainsi l'essentiel de ses exemples, comme un vrai système entraîné sur les attaques déjà observées.

**Pourquoi réutiliser les détecteurs de l'expérience 1 ?** Ils ont été entraînés sur du trafic normal uniquement : **aucune** famille d'attaque n'a été vue, donc toutes sont « inconnues » pour eux. Les réentraîner ne changerait rien et rendrait la comparaison avec l'expérience 1 moins directe.

**Pourquoi un Random Forest comme classifieur supervisé ?** C'est une référence classique en détection d'intrusion, robuste sans réglage fin, et le meilleur compromis performance/temps dans le notebook supervisé précédent. `class_weight="balanced_subsample"` recalcule les poids des classes dans chaque échantillon d'arbre pour qu'aucune classe ne domine l'apprentissage.

**Pourquoi le même seuil (FPR visé) pour le supervisé ?** Avec le seuil par défaut de 0,5, le supervisé aurait un FPR très différent des détecteurs. Comparer des rappels obtenus à des FPR différents n'a pas de sens : il suffit d'abaisser un seuil pour augmenter le rappel. À FPR égal, la comparaison est équitable.

**Pourquoi une PR-AUC « normal vs inconnues » ?** Elle mesure la séparation entre le trafic normal et les seules attaques inconnues, sans que les attaques connues (plus faciles pour le supervisé) n'interviennent. C'est la mesure la plus directe de la capacité zero-day.

**Critère de décision pour H3, fixé à l'avance** : H3 est soutenue si au moins un détecteur d'anomalies a un F1 global plus bas que le supervisé **et** un taux de détection plus élevé que lui sur au moins une famille inconnue.

In [ ]:
UNK = [f for f in UNKNOWN_FAMILIES if f in set(train_df["attack_cat"])]
sup_train = train_part[~train_part["attack_cat"].isin(UNK)]
sup_val = val_df[~val_df["attack_cat"].isin(UNK)]

# Contrôles : aucune famille inconnue dans l'apprentissage ni le réglage
assert not sup_train["attack_cat"].isin(UNK).any() and not sup_val["attack_cat"].isin(UNK).any()
assert test_df["attack_cat"].isin(UNK).any(), "Aucune famille inconnue dans le test"
print("Familles inconnues :", UNK)
print("Familles connues du classifieur :", sorted(set(sup_train["attack_cat"]) - {"Normal"}))

rf_pipe = make_pipeline(RandomForestClassifier(
    n_estimators=200, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE))
t0 = time.perf_counter()
rf_pipe.fit(sup_train[FEATURES], sup_train["label"])
print(f"Random Forest entraîné en {time.perf_counter() - t0:.1f} s sur {len(sup_train)} flux")

rf_val = rf_pipe.predict_proba(sup_val[FEATURES])[:, 1]
rf_thr = threshold_from_normals(rf_val[sup_val["label"].to_numpy() == 0])
rf_test = rf_pipe.predict_proba(X_test)[:, 1]

SUP = "Random Forest (supervisé)"
all_scores = {**test_scores, SUP: rf_test}
all_thr = {**exp1_thr, SUP: rf_thr}
MODELS3 = DETECTORS + [SUP]

In [ ]:
group = np.where(y_test == 0, "Normal", np.where(test_df["attack_cat"].isin(UNK), "Inconnue", "Connue"))
mask_unk_eval = (group == "Normal") | (group == "Inconnue")

rows = {}
for name in MODELS3:
    s, thr = all_scores[name], all_thr[name]
    pred = s >= thr
    rates = pd.Series(pred).groupby(group).mean()
    glob = detection_metrics(y_test, s, thr)
    rows[name] = {
        "FPR (normaux)": rates.get("Normal", np.nan),
        "Rappel attaques connues": rates.get("Connue", np.nan),
        "Rappel attaques inconnues": rates.get("Inconnue", np.nan),
        "PR-AUC normal vs inconnues": average_precision_score(y_test[mask_unk_eval], s[mask_unk_eval]),
        "ROC-AUC normal vs inconnues": roc_auc_score(y_test[mask_unk_eval], s[mask_unk_eval]),
        "F1 global": glob["F1"], "Précision globale": glob["Précision"], "PR-AUC globale": glob["PR-AUC"],
    }
exp3 = pd.DataFrame(rows).T
save_table(exp3, "exp3_connues_inconnues")
exp3.style.format("{:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
exp3[["Rappel attaques connues", "Rappel attaques inconnues"]].plot(kind="bar", ax=ax, rot=15)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_title(f"Expérience 3 — Détection des attaques connues vs inconnues (FPR visé {TARGET_FPR:.0%})")
plt.tight_layout(); save_fig(fig, "exp3_connues_inconnues"); plt.show()

fam_exp3 = pd.DataFrame({name: family_detection(all_scores[name], all_thr[name]) for name in MODELS3})
fam_exp3.index = [f"{f} ★" if f in UNK else f for f in fam_exp3.index]
fam_exp3 = fam_exp3.loc[sorted(fam_exp3.index, key=lambda f: ("★" not in f, f != "Normal", f))]
save_table(fam_exp3, "exp3_detection_par_famille")

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(fam_exp3.drop(index="Normal", errors="ignore"), annot=True, fmt=".0%", cmap="RdYlGn", vmin=0, vmax=1, cbar=False, ax=ax)
ax.set_title("Taux de détection par famille (★ = inconnue pendant l'apprentissage)")
plt.tight_layout(); save_fig(fig, "exp3_heatmap_familles"); plt.show()

In [ ]:
# Observation automatique pour H3
sup_f1 = exp3.loc[SUP, "F1 global"]
print("OBSERVATION (H3)")
print(f"- {SUP} : F1 global {sup_f1:.1%}, rappel connues {exp3.loc[SUP, 'Rappel attaques connues']:.1%}, "
      f"rappel inconnues {exp3.loc[SUP, 'Rappel attaques inconnues']:.1%}.")
h3_support = False
for name in DETECTORS:
    moins_precis = exp3.loc[name, "F1 global"] < sup_f1
    meilleures = [f for f in UNK if f"{f} ★" in fam_exp3.index and fam_exp3.loc[f"{f} ★", name] > fam_exp3.loc[f"{f} ★", SUP]]
    h3_support |= moins_precis and bool(meilleures)
    print(f"- {name:18s} : F1 global {exp3.loc[name, 'F1 global']:.1%} "
          f"({'inférieur' if moins_precis else 'supérieur ou égal'} au supervisé) ; "
          f"familles inconnues mieux détectées que le supervisé : {meilleures or 'aucune'}.")
print(f"-> H3 {'SOUTENUE' if h3_support else 'NON SOUTENUE'} "
      "(critère : au moins un détecteur moins bon globalement mais meilleur sur au moins une famille inconnue).")
print("Attention : pour les familles avec peu de flux dans le test, une différence de quelques points peut être due au hasard.")

### 7.1 Analyse de l'expérience 3

✍️ *À rédiger avec vos chiffres. Trame proposée :*

**Observation expérimentale** — Rappel connues vs inconnues pour chaque modèle (au même FPR), familles inconnues où un détecteur bat le supervisé, F1 global de chacun.

**Interprétation ML** —
- Le classifieur supervisé apprend une frontière entre le normal et les attaques **qu'il connaît**. Une famille inconnue est détectée seulement si elle ressemble à une famille connue (par exemple *Backdoor* proche de *DoS* ou *Exploits* dans UNSW-NB15). Sinon, il la classe du côté normal.
- Le détecteur d'anomalies ne dépend d'aucune famille : il détecte ce qui s'écarte du normal, connu ou non. Son rappel connues/inconnues doit donc être proche ; c'est sa force en zero-day, et sa faiblesse en précision globale.
- Si le supervisé détecte bien les familles inconnues, c'est le signe qu'elles partagent des caractéristiques avec les familles connues : elles ne sont pas vraiment « nouvelles » du point de vue des variables.

**Interprétation cybersécurité** — En pratique, on ne choisit pas : un SOC combine des détections par signatures/supervisées (précises sur le connu) et des détections comportementales (couverture de l'inconnu, au prix de plus d'alertes à trier). Cette expérience mesure ce que chaque couche apporte.

**Limites** — Une seule combinaison de familles inconnues (le résultat peut changer avec un autre choix) ; peu d'exemples pour certaines familles ; un « zero-day » simulé reste une famille connue du monde réel, générée par le même outil (IXIA) que les autres ; voir Q4.

## 8. Analyse approfondie

### 8.1 Explicabilité : importance par permutation

Méthode : on mélange une variable à la fois et on mesure la baisse de **PR-AUC** (capacité à classer les attaques au-dessus du normal). Plus la baisse est forte, plus le modèle s'appuie sur cette variable. Elle fonctionne aussi bien pour un détecteur d'anomalies (avec son score) que pour le classifieur supervisé, et porte sur les variables d'origine, donc les noms sont exacts.

Calcul sur un échantillon de **validation** (pas le test). On compare le détecteur retenu en 5.1 et le Random Forest de l'expérience 3.

#### Pourquoi la permutation plutôt que SHAP ou les importances des arbres ?

- **Une seule méthode pour tous les modèles** : les importances intégrées n'existent que pour les arbres (et pas pour LOF ni One-Class SVM). Comparer un détecteur et le supervisé exige la même méthode.
- **SHAP** donne des explications plus fines (par flux), mais il n'est rapide et exact que pour les arbres ; pour LOF ou One-Class SVM, il faut `KernelExplainer`, très coûteux sur des milliers de flux. La permutation est un compromis raisonnable dans le temps imparti.
- **Variables d'origine** : on mélange une colonne brute (par exemple `proto`) avant le prétraitement, donc l'importance porte sur la variable telle qu'un analyste la connaît, et non sur une colonne one-hot.
- **Calcul sur la validation** : l'explication ne doit pas reposer sur le test, réservé à la mesure.
- **Score PR-AUC** : cohérent avec le critère principal du notebook.
- **3 répétitions, 5 000 flux** : compromis entre stabilité et temps de calcul (LOF et One-Class SVM doivent rescorer l'échantillon pour chaque variable et chaque répétition).

**Ce que la méthode ne dit pas** : une variable importante est une variable **utilisée** par le modèle. Cela ne prouve pas qu'elle **cause** l'attaque. Et si deux variables sont très corrélées (`sbytes` et `sload`), mélanger l'une ne gêne pas le modèle qui utilise encore l'autre : leur importance individuelle est alors sous-estimée.

In [ ]:
def ap_anomaly(estimator, X, y):
    return average_precision_score(y, -estimator.score_samples(X))

expl_n = 2_000 if FAST_MODE else 5_000
expl = val_df.sample(n=min(expl_n, len(val_df)), random_state=RANDOM_STATE)
expl_sup = expl[~expl["attack_cat"].isin(UNK)]

perm_anom = permutation_importance(exp1_pipes[best_name], expl[FEATURES], expl["label"],
                                   scoring=ap_anomaly, n_repeats=3, random_state=RANDOM_STATE)
perm_sup = permutation_importance(rf_pipe, expl_sup[FEATURES], expl_sup["label"],
                                  scoring="average_precision", n_repeats=3, random_state=RANDOM_STATE)

importance = pd.DataFrame({best_name: perm_anom.importances_mean, SUP: perm_sup.importances_mean},
                          index=FEATURES)
save_table(importance.sort_values(best_name, ascending=False), "importance_permutation")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, color in zip(axes, [best_name, SUP], ["darkorange", "steelblue"]):
    top = importance[col].sort_values(ascending=False).head(12)[::-1]
    ax.barh(top.index, top.values, color=color)
    ax.set_title(f"{col}"); ax.set_xlabel("Baisse de PR-AUC quand la variable est mélangée")
plt.suptitle("Importance par permutation (validation)")
plt.tight_layout(); save_fig(fig, "importance_permutation"); plt.show()

✍️ *À compléter* — Les deux modèles s'appuient-ils sur les mêmes variables ? Un détecteur d'anomalies qui dépend surtout des volumes (octets, paquets) ou des durées est plus facile à justifier auprès d'un analyste. Une forte dépendance aux TTL est suspecte (voir 8.2). Rappel : une importance élevée indique une **association** utilisée par le modèle, pas une cause de l'attaque.

### 8.2 Robustesse : sans les variables de TTL

Dans UNSW-NB15, les TTL (`sttl`, `dttl`, `ct_state_ttl`) diffèrent souvent entre machines d'attaque et machines normales, à cause de la façon dont le banc de test a été construit. C'est un **artefact de simulation** : un modèle qui s'en sert risque de ne pas fonctionner sur un autre réseau. On réentraîne Isolation Forest sans ces variables et on compare.

**Pourquoi seulement Isolation Forest ?** La question porte sur le **dataset** (les performances dépendent-elles d'un artefact ?), pas sur la comparaison des modèles. Un seul détecteur suffit à y répondre, et Isolation Forest est le plus rapide à réentraîner. Refaire toutes les expériences sans TTL est proposé en piste.

**Pourquoi ces trois variables ?** `sttl` et `dttl` sont les TTL source et destination ; `ct_state_ttl` est construite à partir d'eux. Retirer seulement les deux premières laisserait l'information passer par la troisième.

In [ ]:
TTL_COLS = [c for c in ["sttl", "dttl", "ct_state_ttl"] if c in FEATURES]
cols_no_ttl = ([c for c in CONT_COLS if c not in TTL_COLS], [c for c in DISC_COLS if c not in TTL_COLS], CAT_COLS)

if_no_ttl = make_pipeline(make_detectors()["Isolation Forest"], cols_no_ttl).fit(X_fit)
thr_no_ttl = threshold_from_normals(anomaly_score(if_no_ttl, X_val_normal))
s_no_ttl = anomaly_score(if_no_ttl, X_test)

robust = pd.DataFrame({
    "Avec TTL": detection_metrics(y_test, test_scores["Isolation Forest"], exp1_thr["Isolation Forest"]),
    "Sans TTL": detection_metrics(y_test, s_no_ttl, thr_no_ttl),
}).T[PCT_COLS]
save_table(robust, "robustesse_ttl")
display(robust.style.format("{:.1%}"))

fam_rob = pd.DataFrame({
    "Avec TTL": family_detection(test_scores["Isolation Forest"], exp1_thr["Isolation Forest"]),
    "Sans TTL": family_detection(s_no_ttl, thr_no_ttl)})
fam_rob["Δ"] = fam_rob["Sans TTL"] - fam_rob["Avec TTL"]
fam_rob.sort_values("Δ").style.format("{:+.1%}", subset=["Δ"]).format("{:.1%}", subset=["Avec TTL", "Sans TTL"])

✍️ *À compléter* — Si les performances chutent fortement sans TTL, une partie de la détection repose sur l'artefact du banc de test, ce qui limite la généralisation à un vrai réseau. Si elles restent proches, le modèle exploite des signaux plus généraux.

## 9. Questions d'analyse obligatoires

La cellule suivante rassemble les **chiffres mesurés** sur lesquels appuyer chaque réponse.

In [ ]:
print("=" * 70, "\nSYNTHÈSE CHIFFRÉE POUR Q1–Q4\n" + "=" * 70)

# Q1 — anomalies qui ne sont pas des attaques
fp_best, n_norm = int(exp1_test.loc[best_name, "FP"]), int((y_test == 0).sum())
prec_best = exp1_test.loc[best_name, "Précision"]
print(f"\n[Q1] {best_name} : {fp_best} flux normaux signalés comme anormaux sur {n_norm} "
      f"(FPR {fp_best / n_norm:.1%}). Précision = {prec_best:.1%} : "
      f"{1 - prec_best:.1%} des alertes ne sont PAS des attaques.")

# Q2 — attaques qui ressemblent au trafic normal
worst = fam_exp1.drop(index="Normal", errors="ignore")[best_name].sort_values().head(3)
print(f"\n[Q2] Familles les moins détectées par {best_name} : "
      + ", ".join(f"{f} ({v:.0%})" for f, v in worst.items()))
worst_sup = fam_exp3.drop(index="Normal", errors="ignore")[SUP].sort_values().head(3)
print("     Familles les moins détectées par le supervisé : "
      + ", ".join(f"{f} ({v:.0%})" for f, v in worst_sup.items()))

# Q3 — FP acceptables : traduction en alertes / jour
q3 = pd.DataFrame({
    "Rappel (test)": recall_at.loc[best_name],
    "FPR (test)": fpr_at.loc[best_name],
})
q3["Fausses alertes / jour"] = (q3["FPR (test)"] * VOLUME_NORMAL_PAR_JOUR).round().astype(int)
q3.index = [f"FPR visé {f:.0%}" for f in q3.index]
save_table(q3, "q3_alertes_par_jour")
vol = f"{VOLUME_NORMAL_PAR_JOUR:,}".replace(",", " ")
print(f"\n[Q3] {best_name}, pour {vol} flux normaux/jour (hypothèse) :")
display(q3.style.format({"Rappel (test)": "{:.1%}", "FPR (test)": "{:.2%}", "Fausses alertes / jour": "{:,}"}))

# Q4 — portée de la simulation zero-day
print(f"[Q4] Zero-day simulé sur {len(UNK)} famille(s) : {UNK}, "
      f"soit {int((group == 'Inconnue').sum())} flux de test. "
      f"Rappel inconnues : de {exp3.loc[DETECTORS, 'Rappel attaques inconnues'].min():.1%} à "
      f"{exp3.loc[DETECTORS, 'Rappel attaques inconnues'].max():.1%} selon le détecteur.")

### Q1. Une anomalie est-elle nécessairement une attaque ?

**Non.** Un détecteur d'anomalies signale ce qui est **rare ou inhabituel** par rapport au trafic d'apprentissage, pas ce qui est malveillant.
- *Observation* : ✍️ citer le nombre de faux positifs et la précision du modèle retenu (synthèse ci-dessus), et les services/états qui concentrent les fausses alertes (5.6).
- *Interprétation ML* : le modèle n'a aucune notion d'intention ; un flux légitime mais rare (nouvel outil, sauvegarde volumineuse, service peu utilisé) est aussi éloigné du normal qu'une attaque.
- *Interprétation cyber* : chaque anomalie doit être **triée** par un analyste ou enrichie (réputation des IP, contexte de l'hôte) avant d'être traitée comme incident. Inversement, une attaque discrète n'est pas forcément une anomalie (voir Q2).

### Q2. Quelles attaques ressemblent trop au trafic normal ?

- *Observation* : ✍️ citer les familles au plus faible taux de détection (synthèse et tables 5.5 / 7).
- *Interprétation ML* : dans UNSW-NB15, des familles comme *Exploits*, *Fuzzers*, *Analysis* ou *Backdoor* produisent souvent des flux courts, de taille modeste, sur des services courants : leurs statistiques de flux tombent dans la zone dense du trafic normal, donc leurs scores chevauchent ceux du normal (5.3).
- *Interprétation cyber* : ce sont les attaques qui imitent un usage légitime au niveau réseau. Les détecter demande d'autres informations : contenu applicatif, séquences de flux d'un même hôte, journaux système.

### Q3. Quel niveau de faux positifs serait acceptable dans un SOC ?

- *Observation* : ✍️ utiliser le tableau Q3 : nombre de fausses alertes par jour et rappel associé pour chaque FPR.
- *Interprétation* : un FPR qui semble faible devient énorme en volume. À 1 % sur 1 million de flux normaux par jour, cela fait 10 000 fausses alertes, bien au-delà de ce qu'une équipe peut traiter (de l'ordre de quelques dizaines à quelques centaines d'alertes par analyste et par jour, selon l'organisation). Le seuil acceptable dépend donc du **volume**, de la **taille de l'équipe** et du **coût d'une attaque manquée**.
- *Conséquence* : un détecteur d'anomalies seul doit fonctionner à un FPR très bas (bien sous 1 %), ou servir à **prioriser** des alertes plutôt qu'à en créer directement, ou être agrégé (alerte sur un hôte qui cumule les anomalies, pas sur chaque flux).
- *Limite* : le volume de 1 million de flux/jour est une hypothèse ; il faut l'adapter au contexte.

### Q4. Pourquoi cette expérience ne suffit-elle pas à prouver la détection de tous les zero-day ?

1. **Échantillon limité** : seules quelques familles ont été testées comme inconnues, avec parfois peu de flux ; une autre combinaison donnerait d'autres résultats.
2. **Pas de vrais zero-day** : les familles « inconnues » sont des attaques connues du monde réel, générées par le même outil (IXIA PerfectStorm) dans le même banc de test que les autres ; elles partagent des artefacts avec elles.
3. **Dataset simulé et daté (2015)** : le trafic normal et les attaques ne représentent pas un réseau d'entreprise actuel ; les TTL montrent des artefacts de simulation (8.2).
4. **Un vrai zero-day peut ressembler au normal** : une attaque conçue pour imiter du trafic légitime (Q2) échappe par construction à un détecteur d'anomalies basé sur des statistiques de flux.
5. **Validation sur un seul dataset** : aucune preuve de généralisation à un autre réseau.

Conclusion : l'expérience montre **si** et **dans quelle mesure** l'approche aide sur des familles absentes de l'apprentissage, dans ce banc de test. Elle ne prouve pas une capacité générale à détecter des attaques inédites.

## 10. Conclusion et limites générales

### Réponse à la question centrale

✍️ *À rédiger à partir des observations automatiques des sections 5.4, 6 et 7 :*
- **H1** : soutenue / non soutenue — chiffres clés.
- **H2** : soutenue / non soutenue — chiffres clés.
- **H3** : soutenue / non soutenue — chiffres clés.
- Réponse nuancée : on peut détecter **une partie** des attaques inconnues sans exemples, au prix d'un taux de fausses alertes à quantifier ; l'approche complète une détection supervisée plutôt qu'elle ne la remplace.

### Limites générales de l'étude
- Dataset de laboratoire (UNSW-NB15, 2015), avec des artefacts de simulation (TTL) ;
- entraînement des détecteurs sur un échantillon de `N_FIT` flux pour des raisons de coût de calcul (LOF et One-Class SVM) ;
- hyperparamètres fixés a priori, non optimisés ;
- seuils fixés sur une validation propre et issue de la même distribution que l'entraînement ;
- analyse au niveau du flux isolé, sans contexte temporel ni par hôte ;
- incertitude élevée sur les familles rares.

### Pistes
- Régler les hyperparamètres sur la validation ; tester des autoencodeurs comme 4e détecteur ;
- répéter l'expérience 3 avec plusieurs combinaisons de familles inconnues ;
- valider sur un autre dataset (CIC-IDS2017, ToN-IoT) ;
- agréger les anomalies par hôte et par fenêtre de temps pour réduire les fausses alertes.

### Utilisation de l'IA
✍️ *À compléter dans le rapport : outils utilisés, pour quelles tâches (structure du code, explications d'API, rédaction), et ce qui a été vérifié par le groupe.*

Les tableaux et figures sont enregistrés dans le dossier `results/` pour le rapport et la présentation.